# Multi-Source Visual Search: Vanilla VLM Baseline Benchmark (Google Colab Pro)

Evaluates baseline Vision-Language Models on `200_samples.json` with candidate pool `N_20` by default.
Outputs are atomically appended per sample to JSONL in Google Drive to support uninterrupted auto-resumption.

In [ ]:
# Cell 1: Mount Google Drive & Environment Configuration
import os
import sys
from google.colab import drive

# EDIT THIS PATH: Root directory in your Google Drive
DRIVE_ROOT = "/content/drive/MyDrive/cvpr_research"
drive.mount("/content/drive")

DATASET_PATH = os.path.join(DRIVE_ROOT, "dataset", "200_samples.json")
IMAGE_DIR = os.path.join(DRIVE_ROOT, "dataset", "images")
MODELS_DIR = os.path.join(DRIVE_ROOT, "models")
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "results_baseline")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Optional: Retrieve Hugging Face token from Colab Secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("Loaded HF_TOKEN from Colab Secrets.")
except Exception:
    print("No HF_TOKEN secret found; proceeding with public access.")

print(f"Dataset Path : {DATASET_PATH}")
print(f"Image Dir    : {IMAGE_DIR}")
print(f"Output Dir   : {OUTPUT_DIR}")

In [ ]:
# Cell 2: Install Dependencies
!pip install -q transformers accelerate torchvision pillow sentencepiece

In [ ]:
# Cell 3: Clone MSSearch Repository (or Set Local Path)
REPO_DIR = "/content/MSSearch"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/your-org/MSSearch.git {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
    sys.path.insert(0, os.path.join(REPO_DIR, "experiments", "baseline"))
print("Repository path configured.")

In [ ]:
# Cell 4: Launch Evaluation
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"  # Choose any from the 17 benchmark models
POOL_MODE = "N_20"                         # Default candidate pool size
EVAL_STRATEGY = "multi_image"              # 'multi_image' or 'one_by_one'

!python {REPO_DIR}/experiments/baseline/eval_runner.py \
    --model_path "{MODEL_NAME}" \
    --dataset_path "{DATASET_PATH}" \
    --image_dir "{IMAGE_DIR}" \
    --output_dir "{OUTPUT_DIR}" \
    --pool_mode "{POOL_MODE}" \
    --eval_strategy "{EVAL_STRATEGY}" \
    --resume